# R2 Bucket Inspector

Using boto3, authenticate and reach the `cytemaps` R2 bucket and inspect its contents.

In [1]:
import os
from pathlib import Path

import boto3
from dotenv import load_dotenv

load_dotenv(dotenv_path=Path("..") / ".env")

BUCKET = os.environ["BUCKET"]
ENDPOINT_URL = os.environ["ENDPOINT_URL"]

# Test file name
KEY = "test.txt"

client = boto3.client(
    "s3",
    endpoint_url=ENDPOINT_URL,
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
)

print(f"Endpoint : {ENDPOINT_URL}")
print(f"Bucket   : {BUCKET}")

Endpoint : https://8a09aa0b1ce3614588bfcd221d8c183e.eu.r2.cloudflarestorage.com
Bucket   : cytemaps


## 1. List all objects

In [2]:
paginator = client.get_paginator("list_objects_v2")
objects = []
for page in paginator.paginate(Bucket=BUCKET):
    objects.extend(page.get("Contents") or [])

if not objects:
    print("Bucket is empty.")
else:
    print(f"{len(objects)} object(s):")
    for obj in objects:
        size_mb = obj["Size"] / 1024 / 1024
        uploaded = obj["LastModified"].strftime("%Y-%m-%d %H:%M")
        print(f"  {obj['Key']}  ({size_mb:.1f} MB)  {uploaded}")

1593 object(s):
  annotation_pipeline_20260506_121141/SRX12708356_annotated.h5ad  (14.5 MB)  2026-05-06 10:36
  annotation_pipeline_20260507_105737/SRX12708356_clustered.h5ad  (11.4 MB)  2026-05-07 09:02
  annotation_pipeline_20260507_190621/ERX11662369_clustered.h5ad  (831.2 MB)  2026-05-07 17:18
  annotation_pipeline_20260507_190621/SRX19886039_clustered.h5ad  (961.7 MB)  2026-05-07 17:22
  annotation_pipeline_20260508_115128/ERX11662369_clustered.h5ad  (831.2 MB)  2026-05-08 09:52
  annotation_pipeline_20260508_115128/SRX19886039_clustered.h5ad  (961.7 MB)  2026-05-08 09:52
  annotation_pipeline_20260508_121117/ERX11662369_clustered.h5ad  (831.2 MB)  2026-05-08 10:12
  annotation_pipeline_20260508_121117/SRX19886039_clustered.h5ad  (961.7 MB)  2026-05-08 10:12
  annotation_pipeline_20260509_204944/SRX12366723_clustered.h5ad  (128.0 MB)  2026-05-09 18:50
  annotation_pipeline_20260509_204944/SRX13198730_clustered.h5ad  (637.3 MB)  2026-05-09 18:50
  annotation_pipeline_20260509_20494

## 2. Filter by prefix

In [3]:
PREFIX = "annotation_pipeline_20260509_210520/"

paginator = client.get_paginator("list_objects_v2")
filtered = []
for page in paginator.paginate(Bucket=BUCKET, Prefix=PREFIX):
    filtered.extend(page.get("Contents") or [])

if not filtered:
    print(f"No objects found under '{PREFIX}'.")
else:
    print(f"{len(filtered)} object(s) under '{PREFIX}':")
    for obj in filtered:
        size_mb = obj["Size"] / 1024 / 1024
        print(f"  {obj['Key']}  ({size_mb:.1f} MB)")

3 object(s) under 'annotation_pipeline_20260509_210520/':
  annotation_pipeline_20260509_210520/SRX12366723_clustered.h5ad  (128.0 MB)
  annotation_pipeline_20260509_210520/SRX13198730_clustered.h5ad  (637.3 MB)
  annotation_pipeline_20260509_210520/SRX17412841_clustered.h5ad  (327.5 MB)


## 3. Show unique prefixes

In [4]:
# List all unique file prefixes (i.e., parts before the first '/')
unique_prefixes = set()
for obj in objects:
    key = obj["Key"]
    prefix = key.split("/", 1)[0] if "/" in key else key
    unique_prefixes.add(prefix)
if not unique_prefixes:
    print("No prefixes found.")
else:
    print(f"Unique prefixes ({len(unique_prefixes)}):")
    for p in sorted(unique_prefixes):
        print(f"- {p}")

Unique prefixes (12):
- annotation_pipeline_20260506_121141
- annotation_pipeline_20260507_105737
- annotation_pipeline_20260507_190621
- annotation_pipeline_20260508_115128
- annotation_pipeline_20260508_121117
- annotation_pipeline_20260509_204944
- annotation_pipeline_20260509_210520
- arc-institute-virtual-cell-atlas
- cluster-validation-2
- clustered_20260509
- cytetype
- test.txt


## 4. Delete files with a given prefix

In [ ]:
# TODO: Implement

In [ ]:
from pathlib import Path

p = Path().resolve().parent / "scripts" / "cluster_validation" / "cell_type_metrics.py"
p.parent

## 5. Upload a test file

In [ ]:
# import io

# body = b"hello from r2_test.ipynb"

# client.put_object(Bucket=BUCKET, Key=KEY, Body=body)
# print(f"Uploaded '{KEY}' ({len(body)} bytes) to r2://{BUCKET}/{KEY}")

## 6. Read the test file back

In [ ]:
response = client.get_object(Bucket=BUCKET, Key=KEY)
content = response["Body"].read().decode()
print(f"Contents of '{KEY}': {content!r}")

## 7. Download the test file to repo root

In [ ]:
from shared.repo import REPO_ROOT

KEY = "arc-institute-virtual-cell-atlas/scbasecount/2026-01-12/h5ad/GeneFull/Homo_sapiens/SRX22996378.h5ad"

dest = REPO_ROOT / "tmp" / "SRX22996378_from_r2.h5ad"
client.download_file(BUCKET, KEY, str(dest))
print(f"Downloaded '{KEY}' -> {dest}")
# print(f"Contents: {dest.read_text()!r}")